# Table SOM.1 - 16-cell joint distribution of the four hypothesis indicators

Each row is a 4-bit combination `H1 H2a H2b H4` (1 = TRUE). This table reports the count and % within each study for all 16 combinations.

### Satisfying all four conditions

About a quarter of participants in each study satisfy all four conditions at once, far above the 6.25% expected by chance (four independent Bernoulli trials at 0.5).

Input: `data/1_derived/competition/wide_clean.csv` (the cleaned sample from Step 2).

Output: `output/2_analysis/tables/som1_joint_distribution.{csv,md}`.

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / 'run_all.py').exists():
    if ROOT.parent == ROOT:
        raise RuntimeError('Could not find project root (run_all.py)')
    ROOT = ROOT.parent

TABLES = ROOT / 'output' / '2_analysis' / 'tables'
TABLES.mkdir(parents=True, exist_ok=True)
wide = pd.read_csv(ROOT / 'data' / '1_derived' / 'competition' / 'wide_clean.csv')
STUDY_LABELS = {'exp72': 'Study 1', 'exp73': 'Study 2', 'exp58': 'Study 3'}

In [2]:
# Encode each participant's four indicator outcomes as a 4-bit string in
# H1, H2a, H2b, H4 order (e.g. "1101"), then tabulate counts and within-study
# percentages for all 16 possible combinations, plus a Total row.
indicators = ['H1', 'H2a', 'H2b', 'H4']
wide['combo'] = wide[indicators].astype(int).astype(str).agg(''.join, axis=1)

# All 16 combinations, in descending order (1111 down to 0000)
all_combos = [''.join(map(str, [a, b, c, d])) for a in (1, 0) for b in (1, 0) for c in (1, 0) for d in (1, 0)]

rows = []
for combo in all_combos:
    row = {'Combination': combo}
    for s, lab in STUDY_LABELS.items():
        sub = wide[wide['study'] == s]
        n = (sub['combo'] == combo).sum()
        pct = n / len(sub) * 100
        row[f'{lab} N']  = n
        row[f'{lab} %'] = round(pct, 1)
    rows.append(row)
tot = {'Combination': 'Total'}
for s, lab in STUDY_LABELS.items():
    n = (wide['study'] == s).sum()
    tot[f'{lab} N'] = n
    tot[f'{lab} %'] = 100.0
rows.append(tot)
som1 = pd.DataFrame(rows)
print(som1.to_string(index=False))

Combination  Study 1 N  Study 1 %  Study 2 N  Study 2 %  Study 3 N  Study 3 %
       1111        512       26.1        226       22.6         99       23.4
       1110        192        9.8         93        9.3         26        6.1
       1101        354       18.1        191       19.1         57       13.5
       1100         23        1.2         24        2.4          7        1.7
       1011         68        3.5         28        2.8         11        2.6
       1010        162        8.3         76        7.6         16        3.8
       1001         58        3.0         33        3.3         18        4.3
       1000         40        2.0         21        2.1         12        2.8
       0111        273       13.9        158       15.8         75       17.7
       0110        139        7.1         69        6.9         36        8.5
       0101         57        2.9         42        4.2         31        7.3
       0100          4        0.2          1        0.1         

In [3]:
# Highlight: % satisfying all 4 indicators
print('\n% satisfying ALL FOUR (combination 1111):')
for s, lab in STUDY_LABELS.items():
    sub = wide[wide['study'] == s]
    pct = (sub['combo'] == '1111').mean() * 100
    n_at_least_3 = (sub[indicators].sum(axis=1) >= 3).mean() * 100
    print(f'  {lab}: {pct:.1f}%   (>=3 of 4: {n_at_least_3:.1f}%)')


% satisfying ALL FOUR (combination 1111):
  Study 1: 26.1%   (>=3 of 4: 71.4%)
  Study 2: 22.6%   (>=3 of 4: 69.5%)
  Study 3: 23.4%   (>=3 of 4: 63.4%)


In [4]:
csv_path = TABLES / 'som1_joint_distribution.csv'
md_path  = TABLES / 'som1_joint_distribution.md'
som1.to_csv(csv_path, index=False)
with open(md_path, 'w') as f:
    f.write('# Table SOM.1: Joint distribution of the 4 hypothesis indicators\n\n')
    f.write('Combination is `H1 H2a H2b H4` (1 = satisfied).\n\n')
    f.write(som1.to_markdown(index=False))
    f.write('\n')
print(f'wrote {csv_path.relative_to(ROOT)}')
print(f'wrote {md_path.relative_to(ROOT)}')

wrote output\2_analysis\tables\som1_joint_distribution.csv
wrote output\2_analysis\tables\som1_joint_distribution.md
